
# GRAMM MAX-SR V2 — LOCAL
## 4-band Multispectral Super-Resolution • 2.0 m → 1.0 m • 36×36 → 72×72

This notebook implements the **maximum-performance composite architecture** for the exact ×2 GRAMM task.

### Architecture
- residual spectral adapter;
- checkpoint-compatible **EHFNet-MS** spatial backbone;
- axial spectral-spatial Transformer ×2;
- **Mamba ×2** when `mamba_ssm` is installed, with a portable PyTorch SSM fallback;
- Haar high-frequency expert;
- Band-4 specialist at feature level;
- 4-expert per-pixel Mixture-of-Experts;
- native **PixelShuffle ×2**;
- bicubic global residual;
- HR residual-attention refinement;
- Band-4 output specialist;
- 4-band uncertainty head;
- differentiable Gaussian-MTF + exact ×2 area degradation consistency;
- spatial, spectral, frequency, Band-4 and physical losses;
- bounded adaptive loss weighting;
- staged pretrained-backbone fine-tuning;
- AMP, gradient accumulation, gradient clipping, EMA, cosine schedule, auto-resume;
- x8 test-time augmentation;
- ensemble of the best validation checkpoints.

### Existing x2 project reuse
The notebook automatically reuses the prepared exact-x2 dataset under:

`workspace/EHFNet_MS_SR_x2_2m_to_1m/`

Expected pair geometry:

```text
LR : 4 × 36 × 36 @ 2.0 m
GT : 4 × 72 × 72 @ 1.0 m
Footprint: 72 m × 72 m
```

If the prepared arrays do not exist but `data/raw/GRAMM.tif` exists, the notebook can rebuild the exact-x2 dataset locally.

> **Band semantics:** Band 4 receives specialist treatment, but `NDVI` supervision is OFF by default until the physical Red/NIR identities are independently verified.


In [ ]:

# Optional installation cell.
# Install a CUDA-enabled PyTorch build appropriate for your machine FIRST.
# Then uncomment if these packages are missing:
#
# %pip install -U numpy pandas matplotlib tqdm scipy scikit-image rasterio nbformat
#
# Optional true Mamba (recommended mainly on Linux/WSL with a compatible CUDA/PyTorch stack):
# %pip install mamba-ssm
#
# On native Windows the notebook remains runnable without mamba-ssm because
# a portable PyTorch SSM fallback is included.

import os, sys, math, time, json, random, warnings, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm
from scipy.ndimage import gaussian_filter
from skimage.metrics import structural_similarity as skimage_ssim

import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

print("Python :", sys.version.split()[0])
print("OS     :", platform.platform())
print("Torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
print("Device :", DEVICE)

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError("CUDA GPU is required for this MAX-SR training notebook.")


## 1) Configuration and local project paths

In [ ]:

# Set this to your project folder only if auto-detection does not find it.
# Example:
# PROJECT_ROOT_OVERRIDE = Path(r"F:\EHFNet_MS_LOCAL_500_PROJECT")
PROJECT_ROOT_OVERRIDE = None

CWD = Path.cwd().resolve()

if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).resolve()
else:
    candidates = [CWD]
    if CWD.name.lower() == "notebooks":
        candidates.insert(0, CWD.parent)
    candidates += list(CWD.parents)

    PROJECT_ROOT = None
    for p in candidates:
        if (p / "workspace" / "EHFNet_MS_SR_x2_2m_to_1m").exists():
            PROJECT_ROOT = p
            break
        if (p / "data" / "raw" / "GRAMM.tif").exists():
            PROJECT_ROOT = p
            break

    if PROJECT_ROOT is None:
        PROJECT_ROOT = CWD

SOURCE = PROJECT_ROOT / "data" / "raw" / "GRAMM.tif"

ROOT = PROJECT_ROOT / "workspace" / "EHFNet_MS_SR_x2_2m_to_1m"
GT_DIR = ROOT / "gt_1m"
LR_DIR = ROOT / "lrms_2m"
PATCH_DIR = ROOT / "patches_72_36"
REPORT_DIR = ROOT / "reports"

GT_PATH = GT_DIR / "GRAMM_GT_1.0m_4B_Float32.tif"
LR_PATH = LR_DIR / "GRAMM_LRMS_2.0m_4B_Float32.tif"
MANIFEST = PATCH_DIR / "GRAMM_x2_patch_manifest.csv"
NORM_JSON = REPORT_DIR / "train_only_normalization_x2.json"

for d in [GT_DIR, LR_DIR, PATCH_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------- Exact task geometry ----------------
GT_RES_M = 1.0
LR_RES_M = 2.0
SCALE = 2
HR_PATCH = 72
LR_PATCH = 36
FOOTPRINT_M = 72.0

assert HR_PATCH == LR_PATCH * SCALE
assert HR_PATCH * GT_RES_M == LR_PATCH * LR_RES_M == FOOTPRINT_M

# ---------------- Data preparation ----------------
AUTO_PREPARE_IF_MISSING = True
REBUILD_RASTERS = False
REBUILD_PATCHES = False

TRAIN_STRIDE_HR = 36
EVAL_STRIDE_HR = 72
SPLIT_BUFFER_HR = 72

# Existing x2 project assumption. Replace with a measured per-band MTF/PSF later if available.
LR_TOTAL_MTF_NYQ = 0.30

# ---------------- Architecture ----------------
BACKBONE_C = 48       # Keep 48 for compatibility with the existing x2 EHFNet-MS checkpoint.
FUSION_C = 128
NUM_EHF_BLOCKS = 6
NUM_TRANSFORMER_BLOCKS = 2
NUM_MAMBA_BLOCKS = 2
TRANSFORMER_HEADS = 8
HR_REFINE_BLOCKS = 4

SPECIAL_BAND_INDEX = 3   # zero-based => Band 4
ENABLE_BAND4_SPECIALIST = True

# NDVI remains disabled until Red/NIR physical identities are verified.
ENABLE_NDVI_LOSS = False
RED_BAND_INDEX = 2
NIR_BAND_INDEX = 3

# ---------------- Training ----------------
EPOCHS = 1000

# Stable high-performance default for a 48 GB class GPU.
# Increase to 48/64 if your actual peak VRAM leaves substantial headroom.
BATCH_SIZE = 32
ACCUM_STEPS = 2

# NUM_WORKERS=0 is robust inside Windows/Jupyter.
# Raise to 2-4 after confirming the local environment is stable.
NUM_WORKERS = 0

BASE_LR = 2e-4
BACKBONE_LR = 5e-5
MIN_LR_RATIO = 0.01
WEIGHT_DECAY = 1e-5

WARMUP_EPOCHS = 10
BACKBONE_FREEZE_EPOCHS = 10
GRAD_CLIP = 1.0

USE_AMP = True
EMA_DECAY = 0.9995
AUTO_RESUME = True
PERIODIC_EVERY = 25

# ---------------- Inference ----------------
USE_TTA_X8 = True
USE_CHECKPOINT_ENSEMBLE = True

# Optional full-scene export after training.
RUN_FULL_SCENE_EXPORT = False
FULL_SCENE_USE_TTA = True
FULL_SCENE_LR_TILE = 36
FULL_SCENE_LR_STRIDE = 24

# ---------------- Output run ----------------
RUN = ROOT / "training" / "MAXSR_V2_COMPOSITE_1000ep"
CKPT = RUN / "checkpoints"
REPORT = RUN / "reports"
FIG = RUN / "figures"

for d in [CKPT, REPORT, FIG]:
    d.mkdir(parents=True, exist_ok=True)

BEST_TOTAL = CKPT / "best_val_total.pt"
BEST_PIXEL = CKPT / "best_val_pixel.pt"
BEST_BAND4 = CKPT / "best_val_band4_rmse.pt"
RESUME = CKPT / "resume_latest.pt"
LAST = CKPT / "epoch_1000_last.pt"

HISTORY = REPORT / "history.csv"
CONFIG_JSON = REPORT / "config.json"
TEST_JSON = REPORT / "test_metrics.json"

# Existing exact-x2 EHFNet checkpoints used to initialize the compatible backbone.
PRETRAIN_CANDIDATES = [
    ROOT / "training" / "EHFNet_MS_x2_500ep_FULLDATA" / "checkpoints" / "best_val_total.pt",
    ROOT / "training" / "EHFNet_MS_x2_500ep_FULLDATA" / "checkpoints" / "best_val_l1.pt",
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SOURCE      :", SOURCE)
print("ROOT        :", ROOT)
print("RUN         :", RUN)
print("Task        : 4×36×36 @2m -> 4×72×72 @1m")


## 2) Auto-prepare the exact ×2 dataset if needed

In [ ]:

def dataset_ready():
    required = [
        MANIFEST,
        NORM_JSON,
        PATCH_DIR / "train_lr_36x36.npy",
        PATCH_DIR / "train_gt_72x72.npy",
        PATCH_DIR / "val_lr_36x36.npy",
        PATCH_DIR / "val_gt_72x72.npy",
        PATCH_DIR / "test_lr_36x36.npy",
        PATCH_DIR / "test_gt_72x72.npy",
    ]
    return all(p.exists() for p in required)

def patch_coords(x0, x1, H, patch, stride):
    coords = []
    if x1 - x0 < patch or H < patch:
        return coords

    for y in range(0, H - patch + 1, stride):
        for x in range(x0, x1 - patch + 1, stride):
            coords.append((y, x))

    return coords

def prepare_x2_dataset():
    if not SOURCE.exists():
        raise FileNotFoundError(
            "Prepared exact-x2 arrays are missing and GRAMM.tif was not found.\n"
            f"Expected source: {SOURCE}"
        )

    print("Preparing exact ×2 dataset from:", SOURCE)

    with rasterio.open(SOURCE) as src:
        assert src.count == 4
        assert src.crs is not None

        src_bounds = src.bounds
        src_crs = src.crs

    physical_w = src_bounds.right - src_bounds.left
    physical_h = src_bounds.top - src_bounds.bottom

    nested_w_m = math.floor(physical_w / LR_RES_M) * LR_RES_M
    nested_h_m = math.floor(physical_h / LR_RES_M) * LR_RES_M

    GT_W = int(round(nested_w_m / GT_RES_M))
    GT_H = int(round(nested_h_m / GT_RES_M))
    LR_W = GT_W // 2
    LR_H = GT_H // 2

    assert GT_W == 2 * LR_W
    assert GT_H == 2 * LR_H

    GT_TRANSFORM = from_origin(
        src_bounds.left, src_bounds.top,
        GT_RES_M, GT_RES_M
    )
    LR_TRANSFORM = from_origin(
        src_bounds.left, src_bounds.top,
        LR_RES_M, LR_RES_M
    )

    # Controlled 1 m reference.
    if REBUILD_RASTERS or not GT_PATH.exists():
        profile = {
            "driver": "GTiff",
            "height": GT_H,
            "width": GT_W,
            "count": 4,
            "dtype": "float32",
            "crs": src_crs,
            "transform": GT_TRANSFORM,
            "compress": "deflate",
            "predictor": 3,
            "tiled": True,
            "BIGTIFF": "IF_SAFER",
        }

        with rasterio.open(SOURCE) as src, rasterio.open(GT_PATH, "w", **profile) as dst:
            for b in range(1, 5):
                arr = np.empty((GT_H, GT_W), dtype=np.float32)

                reproject(
                    source=rasterio.band(src, b),
                    destination=arr,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=GT_TRANSFORM,
                    dst_crs=src_crs,
                    resampling=Resampling.average,
                )

                dst.write(arr, b)

        print("Created GT:", GT_PATH)

    # Gaussian component of the current engineering MTF model.
    aperture_mtf_nyq = 2.0 / math.pi
    gaussian_mtf_nyq = LR_TOTAL_MTF_NYQ / aperture_mtf_nyq

    lr_nyquist = 1.0 / (2.0 * LR_RES_M)

    sigma_m = math.sqrt(
        -math.log(gaussian_mtf_nyq)
        / (2.0 * math.pi**2 * lr_nyquist**2)
    )

    sigma_gt_px = sigma_m / GT_RES_M

    # Exact nested 2 m LR image.
    if REBUILD_RASTERS or not LR_PATH.exists():
        profile = {
            "driver": "GTiff",
            "height": LR_H,
            "width": LR_W,
            "count": 4,
            "dtype": "float32",
            "crs": src_crs,
            "transform": LR_TRANSFORM,
            "compress": "deflate",
            "predictor": 3,
            "tiled": True,
            "BIGTIFF": "IF_SAFER",
        }

        with rasterio.open(GT_PATH) as gt, rasterio.open(LR_PATH, "w", **profile) as lr:
            for b in range(1, 5):
                x = gt.read(b).astype(np.float32)

                blurred = gaussian_filter(
                    x,
                    sigma=sigma_gt_px,
                    mode="reflect",
                    truncate=4.0,
                )

                y = blurred.reshape(
                    LR_H, 2,
                    LR_W, 2
                ).mean(axis=(1, 3)).astype(np.float32)

                lr.write(y, b)

        print("Created LR:", LR_PATH)

    # Spatial split BEFORE patch extraction.
    b1 = int((0.70 * GT_W) // 2 * 2)
    b2 = int((0.85 * GT_W) // 2 * 2)

    regions = {
        "train": (
            0,
            max(0, b1 - SPLIT_BUFFER_HR),
        ),
        "val": (
            min(GT_W, b1 + SPLIT_BUFFER_HR),
            max(0, b2 - SPLIT_BUFFER_HR),
        ),
        "test": (
            min(GT_W, b2 + SPLIT_BUFFER_HR),
            GT_W,
        ),
    }

    split_coords = {
        "train": patch_coords(
            *regions["train"], GT_H,
            HR_PATCH, TRAIN_STRIDE_HR
        ),
        "val": patch_coords(
            *regions["val"], GT_H,
            HR_PATCH, EVAL_STRIDE_HR
        ),
        "test": patch_coords(
            *regions["test"], GT_H,
            HR_PATCH, EVAL_STRIDE_HR
        ),
    }

    counts = {k: len(v) for k, v in split_coords.items()}
    print("Patch counts:", counts)

    if min(counts.values()) == 0:
        raise RuntimeError(
            "One spatial split has no patches. "
            f"Split counts: {counts}"
        )

    rebuild_arrays = (
        REBUILD_PATCHES
        or not all(
            (PATCH_DIR / f"{split}_lr_36x36.npy").exists()
            and (PATCH_DIR / f"{split}_gt_72x72.npy").exists()
            for split in ("train", "val", "test")
        )
    )

    rows = []

    if rebuild_arrays:
        with rasterio.open(GT_PATH) as gt_ds, rasterio.open(LR_PATH) as lr_ds:
            for split, coords in split_coords.items():
                n = len(coords)

                lr_mm = np.lib.format.open_memmap(
                    PATCH_DIR / f"{split}_lr_36x36.npy",
                    mode="w+",
                    dtype=np.float32,
                    shape=(n, 4, LR_PATCH, LR_PATCH),
                )

                gt_mm = np.lib.format.open_memmap(
                    PATCH_DIR / f"{split}_gt_72x72.npy",
                    mode="w+",
                    dtype=np.float32,
                    shape=(n, 4, HR_PATCH, HR_PATCH),
                )

                for i, (y, x) in enumerate(
                    tqdm(coords, desc=f"Extract {split}")
                ):
                    gt_patch = gt_ds.read(
                        window=rasterio.windows.Window(
                            x, y, HR_PATCH, HR_PATCH
                        )
                    ).astype(np.float32)

                    ly = y // 2
                    lx = x // 2

                    lr_patch = lr_ds.read(
                        window=rasterio.windows.Window(
                            lx, ly, LR_PATCH, LR_PATCH
                        )
                    ).astype(np.float32)

                    gt_mm[i] = gt_patch
                    lr_mm[i] = lr_patch

                    rows.append({
                        "split": split,
                        "index": i,
                        "gt_y": y,
                        "gt_x": x,
                        "lr_y": ly,
                        "lr_x": lx,
                    })

                del lr_mm, gt_mm

        pd.DataFrame(rows).to_csv(MANIFEST, index=False)

    elif not MANIFEST.exists():
        for split, coords in split_coords.items():
            for i, (y, x) in enumerate(coords):
                rows.append({
                    "split": split,
                    "index": i,
                    "gt_y": y,
                    "gt_x": x,
                    "lr_y": y // 2,
                    "lr_x": x // 2,
                })
        pd.DataFrame(rows).to_csv(MANIFEST, index=False)

    # Train-only normalization from 1 m training targets.
    if not NORM_JSON.exists() or REBUILD_PATCHES:
        train_gt = np.load(
            PATCH_DIR / "train_gt_72x72.npy",
            mmap_mode="r"
        )

        sums = np.zeros(4, np.float64)
        sums2 = np.zeros(4, np.float64)
        mins = np.full(4, np.inf, np.float64)
        maxs = np.full(4, -np.inf, np.float64)
        count = 0

        for i in tqdm(range(len(train_gt)), desc="Train normalization"):
            a = np.asarray(train_gt[i], dtype=np.float64)
            flat = a.reshape(4, -1)

            sums += flat.sum(1)
            sums2 += (flat**2).sum(1)
            mins = np.minimum(mins, flat.min(1))
            maxs = np.maximum(maxs, flat.max(1))

            count += flat.shape[1]

        mean = sums / count
        var = sums2 / count - mean**2
        std = np.sqrt(np.maximum(var, 1e-12))

        stats = {
            "band_mean": mean.tolist(),
            "band_std": std.tolist(),
            "band_min": mins.tolist(),
            "band_max": maxs.tolist(),
            "band_range": (maxs - mins).tolist(),
            "source": "TRAIN GT 1m patches only",
            "mtf_nyquist_assumption": LR_TOTAL_MTF_NYQ,
            "gaussian_sigma_gt_px": sigma_gt_px,
        }

        NORM_JSON.write_text(
            json.dumps(stats, indent=2),
            encoding="utf-8",
        )

    print("Exact ×2 preparation complete.")

if not dataset_ready():
    if AUTO_PREPARE_IF_MISSING:
        prepare_x2_dataset()
    else:
        raise RuntimeError(
            "Prepared exact-x2 data is missing. "
            "Run the x2 preparation notebook or set AUTO_PREPARE_IF_MISSING=True."
        )

assert dataset_ready()
print("Exact ×2 dataset: READY")


## 3) Dataset and loaders

In [ ]:

TRAIN_STATS = json.loads(
    NORM_JSON.read_text(encoding="utf-8")
)

BAND_MEAN = np.array(
    TRAIN_STATS["band_mean"],
    dtype=np.float32,
).reshape(4, 1, 1)

BAND_STD = np.array(
    TRAIN_STATS["band_std"],
    dtype=np.float32,
).reshape(4, 1, 1)

TRAIN_RANGE = np.array(
    TRAIN_STATS["band_range"],
    dtype=np.float64,
)

MEAN_T = torch.tensor(
    TRAIN_STATS["band_mean"],
    dtype=torch.float32,
    device=DEVICE,
).view(1, 4, 1, 1)

STD_T = torch.tensor(
    TRAIN_STATS["band_std"],
    dtype=torch.float32,
    device=DEVICE,
).view(1, 4, 1, 1)

def augment_pair(lr, gt):
    k = random.randint(0, 3)

    if k:
        lr = np.rot90(lr, k, axes=(1, 2))
        gt = np.rot90(gt, k, axes=(1, 2))

    if random.random() < 0.5:
        lr = lr[:, :, ::-1]
        gt = gt[:, :, ::-1]

    if random.random() < 0.5:
        lr = lr[:, ::-1, :]
        gt = gt[:, ::-1, :]

    return (
        np.ascontiguousarray(lr),
        np.ascontiguousarray(gt),
    )

class NPYPairDataset(Dataset):
    def __init__(self, split, augment=False):
        self.split = split
        self.augment = augment

        self.lr = np.load(
            PATCH_DIR / f"{split}_lr_36x36.npy",
            mmap_mode="r",
        )

        self.gt = np.load(
            PATCH_DIR / f"{split}_gt_72x72.npy",
            mmap_mode="r",
        )

        assert len(self.lr) == len(self.gt)

    def __len__(self):
        return len(self.lr)

    def __getitem__(self, idx):
        lr = np.array(
            self.lr[idx],
            dtype=np.float32,
            copy=True,
        )

        gt = np.array(
            self.gt[idx],
            dtype=np.float32,
            copy=True,
        )

        if self.augment:
            lr, gt = augment_pair(lr, gt)

        lr = (lr - BAND_MEAN) / BAND_STD
        gt = (gt - BAND_MEAN) / BAND_STD

        return {
            "lr": torch.from_numpy(lr),
            "gt": torch.from_numpy(gt),
            "index": idx,
        }

train_ds = NPYPairDataset("train", augment=True)
val_ds = NPYPairDataset("val", augment=False)
test_ds = NPYPairDataset("test", augment=False)

def make_loader(ds, shuffle, batch_size):
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
        drop_last=False,
    )

train_loader = make_loader(
    train_ds, True, BATCH_SIZE
)

val_loader = make_loader(
    val_ds, False, BATCH_SIZE
)

test_loader = make_loader(
    test_ds, False, BATCH_SIZE
)

probe = next(iter(train_loader))

print(
    "Train / Val / Test:",
    len(train_ds),
    len(val_ds),
    len(test_ds),
)

print(
    "LR:", tuple(probe["lr"].shape),
    "GT:", tuple(probe["gt"].shape),
)

assert probe["lr"].shape[1:] == (4, 36, 36)
assert probe["gt"].shape[1:] == (4, 72, 72)


## 4) Differentiable degradation and composite losses

In [ ]:

# Current exact-x2 differentiable physical consistency operator.

aperture_mtf_nyq = 2.0 / math.pi
gaussian_mtf_nyq = (
    LR_TOTAL_MTF_NYQ
    / aperture_mtf_nyq
)

lr_nyquist = (
    1.0
    / (2.0 * LR_RES_M)
)

sigma_m = math.sqrt(
    -math.log(gaussian_mtf_nyq)
    / (
        2.0
        * math.pi**2
        * lr_nyquist**2
    )
)

sigma_gt_px = (
    sigma_m
    / GT_RES_M
)

def gaussian_kernel_1d(
    sigma,
    truncate=4.0,
):
    radius = int(
        math.ceil(
            truncate * sigma
        )
    )

    x = torch.arange(
        -radius,
        radius + 1,
        dtype=torch.float32,
    )

    k = torch.exp(
        -0.5
        * (x / sigma)**2
    )

    return (
        k / k.sum(),
        radius,
    )

GAUSS_K, GAUSS_RADIUS = (
    gaussian_kernel_1d(
        sigma_gt_px
    )
)

def blur_hr(x):
    c = x.shape[1]

    k = GAUSS_K.to(
        x.device,
        x.dtype,
    )

    kh = k.view(
        1, 1, 1, -1
    ).repeat(
        c, 1, 1, 1
    )

    kv = k.view(
        1, 1, -1, 1
    ).repeat(
        c, 1, 1, 1
    )

    x = F.conv2d(
        F.pad(
            x,
            (
                GAUSS_RADIUS,
                GAUSS_RADIUS,
                0,
                0,
            ),
            mode="reflect",
        ),
        kh,
        groups=c,
    )

    x = F.conv2d(
        F.pad(
            x,
            (
                0,
                0,
                GAUSS_RADIUS,
                GAUSS_RADIUS,
            ),
            mode="reflect",
        ),
        kv,
        groups=c,
    )

    return x

def degradation_operator(x):
    return F.avg_pool2d(
        blur_hr(
            x.float()
        ),
        kernel_size=2,
        stride=2,
    )

def denorm(x):
    return (
        x.float()
        * STD_T
        + MEAN_T
    )

def charbonnier(
    x,
    eps=1e-3,
):
    return torch.sqrt(
        x * x
        + eps * eps
    ).mean()

def sobel_xy(x):
    c = x.shape[1]

    kx = torch.tensor(
        [
            [-1, 0, 1],
            [-2, 0, 2],
            [-1, 0, 1],
        ],
        dtype=x.dtype,
        device=x.device,
    ).view(
        1, 1, 3, 3
    ).repeat(
        c, 1, 1, 1
    )

    ky = torch.tensor(
        [
            [-1, -2, -1],
            [0, 0, 0],
            [1, 2, 1],
        ],
        dtype=x.dtype,
        device=x.device,
    ).view(
        1, 1, 3, 3
    ).repeat(
        c, 1, 1, 1
    )

    return (
        F.conv2d(
            x,
            kx,
            padding=1,
            groups=c,
        ),
        F.conv2d(
            x,
            ky,
            padding=1,
            groups=c,
        ),
    )

def edge_loss(
    pred,
    gt,
):
    px, py = sobel_xy(pred)
    gx, gy = sobel_xy(gt)

    return 0.5 * (
        charbonnier(px - gx)
        + charbonnier(py - gy)
    )

def frequency_loss(
    pred,
    gt,
):
    pf = torch.fft.rfft2(
        pred.float(),
        norm="ortho",
    )

    gf = torch.fft.rfft2(
        gt.float(),
        norm="ortho",
    )

    pa = torch.log1p(
        torch.abs(pf)
    )

    ga = torch.log1p(
        torch.abs(gf)
    )

    H, W = pred.shape[-2:]

    fy = torch.fft.fftfreq(
        H,
        device=pred.device,
    ).abs().view(
        H, 1
    )

    fx = torch.fft.rfftfreq(
        W,
        device=pred.device,
    ).abs().view(
        1, -1
    )

    radius = torch.sqrt(
        fy * fy
        + fx * fx
    )

    weight = (
        radius
        / radius.max().clamp_min(1e-8)
    ).pow(1.5)

    return (
        (pa - ga).abs()
        * weight
    ).mean()

def sam_loss(
    pred,
    gt,
    eps=1e-8,
):
    p = denorm(pred)
    g = denorm(gt)

    dot = (
        p * g
    ).sum(1)

    pn = torch.sqrt(
        (p * p).sum(1).clamp_min(eps)
    )

    gn = torch.sqrt(
        (g * g).sum(1).clamp_min(eps)
    )

    cos = (
        dot
        / (pn * gn).clamp_min(eps)
    ).clamp(
        -1 + 1e-6,
        1 - 1e-6,
    )

    return torch.acos(
        cos
    ).mean()

def interband_loss(
    pred,
    gt,
):
    values = []

    for i in range(4):
        for j in range(
            i + 1,
            4,
        ):
            values.append(
                charbonnier(
                    (
                        pred[:, i:i+1]
                        - pred[:, j:j+1]
                    )
                    -
                    (
                        gt[:, i:i+1]
                        - gt[:, j:j+1]
                    )
                )
            )

    return torch.stack(
        values
    ).mean()

def ndvi_loss(
    pred,
    gt,
    eps=1e-6,
):
    p = denorm(pred)
    g = denorm(gt)

    pr = p[
        :,
        RED_BAND_INDEX:RED_BAND_INDEX+1,
    ]

    pn = p[
        :,
        NIR_BAND_INDEX:NIR_BAND_INDEX+1,
    ]

    gr = g[
        :,
        RED_BAND_INDEX:RED_BAND_INDEX+1,
    ]

    gn = g[
        :,
        NIR_BAND_INDEX:NIR_BAND_INDEX+1,
    ]

    pndvi = (
        (pn - pr)
        / (pn + pr + eps)
    )

    gndvi = (
        (gn - gr)
        / (gn + gr + eps)
    )

    return charbonnier(
        pndvi - gndvi
    )

BASE_LOSS_WEIGHTS = {
    "band4": 1.50,
    "sam": 0.10,
    "bias": 0.05,
    "inter": 0.05,
    "edge": 0.05,
    "freq": 0.05,
    "dc": 0.10,
    "unc": 0.02,
    "gate": 0.003,
    "ndvi": (
        0.05
        if ENABLE_NDVI_LOSS
        else 0.0
    ),
}

print(
    "Gaussian sigma at 1m GT:",
    sigma_gt_px,
    "pixels",
)


## 5) MAX-SR V2 architecture

In [ ]:

class DepthwisePointwise(nn.Module):
    def __init__(
        self,
        c,
        dilation=1,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(
                c,
                c,
                3,
                padding=dilation,
                dilation=dilation,
                groups=c,
            ),
            nn.GELU(),

            nn.Conv2d(
                c,
                c,
                1,
            ),
            nn.GELU(),
        )

    def forward(self, x):
        return self.net(x)


class EdgeEnhancementBlockMS(nn.Module):
    def __init__(
        self,
        c,
        res_scale=0.10,
    ):
        super().__init__()

        self.res_scale = float(
            res_scale
        )

        self.wide = nn.Sequential(
            nn.Conv2d(
                c,
                2 * c,
                1,
            ),
            nn.GELU(),

            nn.Conv2d(
                2 * c,
                2 * c,
                3,
                padding=1,
            ),
            nn.GELU(),

            nn.Conv2d(
                2 * c,
                c,
                1,
            ),
        )

        self.edge = nn.Sequential(
            nn.Conv2d(
                c,
                c,
                3,
                padding=1,
                groups=c,
                bias=False,
            ),
            nn.Conv2d(
                c,
                c,
                1,
            ),
            nn.GELU(),
        )

        lap = torch.tensor(
            [
                [0., -1., 0.],
                [-1., 4., -1.],
                [0., -1., 0.],
            ]
        )

        with torch.no_grad():
            self.edge[0].weight.zero_()

            self.edge[0].weight[
                :, 0, :, :
            ].copy_(lap)

    def forward(self, x):
        return (
            x
            + self.res_scale
            * (
                self.wide(x)
                + self.edge(x)
            )
        )


class MultiScaleAttentionMS(nn.Module):
    def __init__(
        self,
        c,
    ):
        super().__init__()

        self.b1 = DepthwisePointwise(
            c,
            1,
        )

        self.b2 = DepthwisePointwise(
            c,
            2,
        )

        self.b3 = DepthwisePointwise(
            c,
            3,
        )

        self.mix = nn.Conv2d(
            3 * c,
            c,
            1,
        )

        h = max(
            8,
            c // 8,
        )

        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),

            nn.Conv2d(
                c,
                h,
                1,
            ),
            nn.GELU(),

            nn.Conv2d(
                h,
                c,
                1,
            ),
            nn.Sigmoid(),
        )

        self.sa = nn.Sequential(
            nn.Conv2d(
                2,
                1,
                7,
                padding=3,
            ),
            nn.Sigmoid(),
        )

    def forward(self, x):
        m = self.mix(
            torch.cat(
                [
                    self.b1(x),
                    self.b2(x),
                    self.b3(x),
                ],
                dim=1,
            )
        )

        ca = self.ca(m)

        sa = self.sa(
            torch.cat(
                [
                    m.mean(
                        1,
                        keepdim=True,
                    ),
                    m.amax(
                        1,
                        keepdim=True,
                    ),
                ],
                dim=1,
            )
        )

        return (
            x
            + m * ca * sa
        )


class EHFBlockMS(nn.Module):
    def __init__(
        self,
        c,
    ):
        super().__init__()

        self.edge = (
            EdgeEnhancementBlockMS(
                c
            )
        )

        self.attn = (
            MultiScaleAttentionMS(
                c
            )
        )

    def forward(self, x):
        return self.attn(
            self.edge(x)
        )


class EHFBackbone(nn.Module):
    # Checkpoint-compatible LR feature extractor from the existing exact-x2 EHFNet-MS.
    def __init__(
        self,
        c=48,
        blocks=6,
    ):
        super().__init__()

        self.shallow = nn.Sequential(
            nn.Conv2d(
                4,
                c,
                3,
                padding=1,
            ),
            nn.GELU(),
        )

        self.body = nn.Sequential(
            *[
                EHFBlockMS(c)
                for _ in range(blocks)
            ]
        )

        self.body_tail = nn.Conv2d(
            c,
            c,
            3,
            padding=1,
        )

    def forward(self, x):
        s = self.shallow(x)

        return (
            self.body_tail(
                self.body(s)
            )
            + s
        )


class SpectralResidualAdapter(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(
                4,
                32,
                1,
            ),
            nn.GELU(),

            nn.Conv2d(
                32,
                32,
                3,
                padding=1,
            ),
            nn.GELU(),

            nn.Conv2d(
                32,
                4,
                1,
            ),
        )

        self.beta_raw = nn.Parameter(
            torch.zeros(())
        )

        nn.init.zeros_(
            self.net[-1].weight
        )
        nn.init.zeros_(
            self.net[-1].bias
        )

    def forward(self, x):
        beta = (
            0.20
            * torch.tanh(
                self.beta_raw
            )
        )

        return (
            x
            + beta
            * self.net(x),
            beta,
        )


class AxialTransformerBlock(nn.Module):
    # Long-range row + column attention on the 36x36 feature map.
    def __init__(
        self,
        c,
        heads=8,
        mlp_ratio=2.0,
    ):
        super().__init__()

        assert c % heads == 0

        self.norm_r = nn.LayerNorm(c)

        self.row_attn = (
            nn.MultiheadAttention(
                c,
                heads,
                batch_first=True,
            )
        )

        self.norm_c = nn.LayerNorm(c)

        self.col_attn = (
            nn.MultiheadAttention(
                c,
                heads,
                batch_first=True,
            )
        )

        hidden = int(
            c * mlp_ratio
        )

        self.ff = nn.Sequential(
            nn.Conv2d(
                c,
                hidden,
                1,
            ),
            nn.GELU(),

            nn.Conv2d(
                hidden,
                c,
                1,
            ),
        )

        self.ff_scale = nn.Parameter(
            torch.tensor(0.1)
        )

    def forward(self, x):
        B, C, H, W = x.shape

        rows = (
            x.permute(
                0, 2, 3, 1
            )
            .reshape(
                B * H,
                W,
                C,
            )
        )

        rn = self.norm_r(
            rows
        )

        rows = (
            rows
            + self.row_attn(
                rn,
                rn,
                rn,
                need_weights=False,
            )[0]
        )

        x = (
            rows.reshape(
                B,
                H,
                W,
                C,
            )
            .permute(
                0, 3, 1, 2
            )
        )

        cols = (
            x.permute(
                0, 3, 2, 1
            )
            .reshape(
                B * W,
                H,
                C,
            )
        )

        cn = self.norm_c(
            cols
        )

        cols = (
            cols
            + self.col_attn(
                cn,
                cn,
                cn,
                need_weights=False,
            )[0]
        )

        x = (
            cols.reshape(
                B,
                W,
                H,
                C,
            )
            .permute(
                0, 3, 2, 1
            )
        )

        return (
            x
            + torch.tanh(
                self.ff_scale
            )
            * self.ff(x)
        )


HAS_TRUE_MAMBA = False

try:
    from mamba_ssm import Mamba as NativeMamba
    HAS_TRUE_MAMBA = True
except Exception:
    NativeMamba = None


class TrueMamba2DBlock(nn.Module):
    def __init__(
        self,
        c,
    ):
        super().__init__()

        self.norm = nn.LayerNorm(c)

        self.mamba = NativeMamba(
            d_model=c,
            d_state=16,
            d_conv=4,
            expand=2,
        )

        self.scale = nn.Parameter(
            torch.tensor(0.1)
        )

    def forward(self, x):
        B, C, H, W = x.shape

        tokens = (
            x.flatten(2)
            .transpose(1, 2)
        )

        tokens = (
            tokens
            + torch.tanh(
                self.scale
            )
            * self.mamba(
                self.norm(tokens)
            )
        )

        return (
            tokens.transpose(
                1, 2
            )
            .reshape(
                B, C, H, W
            )
        )


class FallbackSSM2DBlock(nn.Module):
    # Portable linear-cost state-space-style fallback when mamba_ssm is unavailable.
    def __init__(
        self,
        c,
    ):
        super().__init__()

        self.norm = nn.GroupNorm(
            1,
            c,
        )

        self.in_proj = nn.Conv2d(
            c,
            2 * c,
            1,
        )

        self.dw = nn.Conv2d(
            c,
            c,
            5,
            padding=2,
            groups=c,
        )

        self.mix = nn.Conv2d(
            c,
            c,
            1,
        )

        self.out = nn.Conv2d(
            c,
            c,
            1,
        )

        self.scale = nn.Parameter(
            torch.tensor(0.1)
        )

    def forward(self, x):
        z = self.norm(x)

        u, gate = self.in_proj(
            z
        ).chunk(
            2,
            dim=1,
        )

        u = self.dw(u)

        h1 = (
            torch.cumsum(
                u,
                dim=-1,
            )
            / math.sqrt(
                max(
                    1,
                    u.shape[-1],
                )
            )
        )

        h2 = (
            torch.flip(
                torch.cumsum(
                    torch.flip(
                        u,
                        dims=[-1],
                    ),
                    dim=-1,
                ),
                dims=[-1],
            )
            / math.sqrt(
                max(
                    1,
                    u.shape[-1],
                )
            )
        )

        v1 = (
            torch.cumsum(
                u,
                dim=-2,
            )
            / math.sqrt(
                max(
                    1,
                    u.shape[-2],
                )
            )
        )

        v2 = (
            torch.flip(
                torch.cumsum(
                    torch.flip(
                        u,
                        dims=[-2],
                    ),
                    dim=-2,
                ),
                dims=[-2],
            )
            / math.sqrt(
                max(
                    1,
                    u.shape[-2],
                )
            )
        )

        state = self.mix(
            0.25
            * (
                h1
                + h2
                + v1
                + v2
            )
        )

        y = self.out(
            state
            * torch.sigmoid(
                gate
            )
        )

        return (
            x
            + torch.tanh(
                self.scale
            )
            * y
        )


def make_mamba_block(c):
    if HAS_TRUE_MAMBA:
        return TrueMamba2DBlock(c)

    return FallbackSSM2DBlock(c)


class HaarFrequencyExpert(nn.Module):
    def __init__(
        self,
        c,
    ):
        super().__init__()

        # LH, HL, HH Haar filters.
        kernels = torch.tensor(
            [
                [
                    [1., 1.],
                    [-1., -1.],
                ],
                [
                    [1., -1.],
                    [1., -1.],
                ],
                [
                    [1., -1.],
                    [-1., 1.],
                ],
            ],
            dtype=torch.float32,
        ) * 0.5

        # Group-wise ordering:
        # three high-pass outputs per input channel.
        weight = (
            kernels[
                None, :, :, :
            ]
            .repeat(
                c, 1, 1, 1
            )
            .reshape(
                3 * c,
                1,
                2,
                2,
            )
        )

        self.register_buffer(
            "haar",
            weight,
        )

        self.proj = nn.Sequential(
            nn.Conv2d(
                3 * c,
                c,
                1,
            ),
            nn.GELU(),

            DepthwisePointwise(
                c,
                1,
            ),

            DepthwisePointwise(
                c,
                2,
            ),
        )

    def forward(self, x):
        B, C, H, W = x.shape

        hf = F.conv2d(
            x,
            self.haar.to(
                x.dtype
            ),
            stride=2,
            groups=C,
        )

        hf = F.interpolate(
            hf,
            size=(H, W),
            mode="bilinear",
            align_corners=False,
        )

        return self.proj(hf)


class Band4Expert(nn.Module):
    def __init__(
        self,
        c,
    ):
        super().__init__()

        self.band = nn.Sequential(
            nn.Conv2d(
                1,
                32,
                3,
                padding=1,
            ),
            nn.GELU(),

            nn.Conv2d(
                32,
                32,
                3,
                padding=1,
            ),
            nn.GELU(),
        )

        self.fuse = nn.Sequential(
            nn.Conv2d(
                c + 32,
                c,
                1,
            ),
            nn.GELU(),

            DepthwisePointwise(
                c,
                1,
            ),

            DepthwisePointwise(
                c,
                2,
            ),
        )

    def forward(
        self,
        band4,
        spatial,
    ):
        return self.fuse(
            torch.cat(
                [
                    self.band(
                        band4
                    ),
                    spatial,
                ],
                dim=1,
            )
        )


class MoEFusion(nn.Module):
    def __init__(
        self,
        c,
        n=4,
    ):
        super().__init__()

        self.gate = nn.Sequential(
            nn.Conv2d(
                n * c,
                c,
                1,
            ),
            nn.GELU(),

            nn.Conv2d(
                c,
                c,
                3,
                padding=1,
                groups=c,
            ),
            nn.GELU(),

            nn.Conv2d(
                c,
                n,
                1,
            ),
        )

    def forward(
        self,
        experts,
    ):
        cat = torch.cat(
            experts,
            dim=1,
        )

        gate = torch.softmax(
            self.gate(cat),
            dim=1,
        )

        fused = torch.zeros_like(
            experts[0]
        )

        for i, expert in enumerate(experts):
            fused = (
                fused
                + gate[
                    :, i:i+1
                ]
                * expert
            )

        return (
            fused,
            gate,
        )


class ResidualAttentionHR(nn.Module):
    def __init__(
        self,
        c,
    ):
        super().__init__()

        h = max(
            8,
            c // 8,
        )

        self.body = nn.Sequential(
            nn.Conv2d(
                c,
                c,
                3,
                padding=1,
                groups=c,
            ),
            nn.GELU(),

            nn.Conv2d(
                c,
                c,
                1,
            ),
            nn.GELU(),

            nn.Conv2d(
                c,
                c,
                3,
                padding=1,
                groups=c,
            ),
        )

        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),

            nn.Conv2d(
                c,
                h,
                1,
            ),
            nn.GELU(),

            nn.Conv2d(
                h,
                c,
                1,
            ),
            nn.Sigmoid(),
        )

        self.scale = nn.Parameter(
            torch.tensor(0.1)
        )

    def forward(self, x):
        y = self.body(x)

        return (
            x
            + torch.tanh(
                self.scale
            )
            * y
            * self.ca(y)
        )


class BoundedLossWeightController(nn.Module):
    def __init__(
        self,
        names,
    ):
        super().__init__()

        self.names = list(
            names
        )

        self.raw = nn.ParameterDict(
            {
                name: nn.Parameter(
                    torch.zeros(())
                )
                for name
                in self.names
            }
        )

    def multiplier(
        self,
        name,
    ):
        # exp(±0.35) ~= 0.70 .. 1.42.
        return torch.exp(
            0.35
            * torch.tanh(
                self.raw[name]
            )
        )

    def regularizer(self):
        values = [
            value.square()
            for value
            in self.raw.values()
        ]

        return (
            torch.stack(values).mean()
            if values
            else torch.zeros(
                (),
                device=DEVICE,
            )
        )


class MaxSRV2(nn.Module):
    def __init__(self):
        super().__init__()

        c = FUSION_C

        self.spectral_adapter = (
            SpectralResidualAdapter()
        )

        self.backbone = EHFBackbone(
            BACKBONE_C,
            NUM_EHF_BLOCKS,
        )

        self.spatial_proj = nn.Sequential(
            nn.Conv2d(
                BACKBONE_C,
                c,
                1,
            ),
            nn.GELU(),

            DepthwisePointwise(
                c,
                1,
            ),
        )

        self.transformers = nn.Sequential(
            *[
                AxialTransformerBlock(
                    c,
                    TRANSFORMER_HEADS,
                )
                for _
                in range(
                    NUM_TRANSFORMER_BLOCKS
                )
            ]
        )

        self.mambas = nn.Sequential(
            *[
                make_mamba_block(c)
                for _
                in range(
                    NUM_MAMBA_BLOCKS
                )
            ]
        )

        self.frequency = (
            HaarFrequencyExpert(
                c
            )
        )

        self.band4_expert = (
            Band4Expert(
                c
            )
        )

        self.moe = MoEFusion(
            c,
            4,
        )

        self.up = nn.Sequential(
            nn.Conv2d(
                c,
                4 * c,
                3,
                padding=1,
            ),
            nn.PixelShuffle(2),
            nn.GELU(),
        )

        self.hr_refine = nn.Sequential(
            *[
                ResidualAttentionHR(c)
                for _
                in range(
                    HR_REFINE_BLOCKS
                )
            ]
        )

        self.out4 = nn.Conv2d(
            c,
            4,
            3,
            padding=1,
        )

        self.band4_refine = nn.Sequential(
            nn.Conv2d(
                1 + 3 * c,
                96,
                1,
            ),
            nn.GELU(),

            ResidualAttentionHR(
                96
            ),

            ResidualAttentionHR(
                96
            ),

            nn.Conv2d(
                96,
                1,
                3,
                padding=1,
            ),
        )

        self.alpha_raw = nn.Parameter(
            torch.tensor(-1.0)
        )

        self.unc_head = nn.Sequential(
            nn.Conv2d(
                c,
                64,
                3,
                padding=1,
            ),
            nn.GELU(),

            nn.Conv2d(
                64,
                64 * 4,
                3,
                padding=1,
            ),

            nn.PixelShuffle(2),
            nn.GELU(),

            nn.Conv2d(
                64,
                4,
                3,
                padding=1,
            ),
        )

        self.loss_weight_controller = (
            BoundedLossWeightController(
                BASE_LOSS_WEIGHTS.keys()
            )
        )

        # Start close to bicubic for stable transfer.
        nn.init.zeros_(
            self.out4.weight
        )
        nn.init.zeros_(
            self.out4.bias
        )

        nn.init.zeros_(
            self.band4_refine[-1].weight
        )
        nn.init.zeros_(
            self.band4_refine[-1].bias
        )

    def forward(self, lr):
        x, beta = (
            self.spectral_adapter(
                lr
            )
        )

        f0 = self.backbone(x)

        f_spatial = (
            self.spatial_proj(
                f0
            )
        )

        f_spectral = (
            self.transformers(
                f_spatial
            )
        )

        f_spectral = (
            self.mambas(
                f_spectral
            )
        )

        f_frequency = (
            self.frequency(
                f_spatial
            )
        )

        if ENABLE_BAND4_SPECIALIST:
            f_band4 = (
                self.band4_expert(
                    x[
                        :,
                        SPECIAL_BAND_INDEX:
                        SPECIAL_BAND_INDEX+1,
                    ],
                    f_spatial,
                )
            )
        else:
            f_band4 = f_spatial

        fused, gate = self.moe(
            [
                f_spatial,
                f_spectral,
                f_frequency,
                f_band4,
            ]
        )

        hr = self.up(fused)
        hr = self.hr_refine(hr)

        base = F.interpolate(
            lr,
            scale_factor=2,
            mode="bicubic",
            align_corners=False,
        )

        sr_initial = (
            base
            + self.out4(hr)
        )

        if ENABLE_BAND4_SPECIALIST:
            fn_hr = F.interpolate(
                f_band4,
                scale_factor=2,
                mode="bilinear",
                align_corners=False,
            )

            ff_hr = F.interpolate(
                fused,
                scale_factor=2,
                mode="bilinear",
                align_corners=False,
            )

            b4_in = torch.cat(
                [
                    sr_initial[
                        :,
                        SPECIAL_BAND_INDEX:
                        SPECIAL_BAND_INDEX+1,
                    ],
                    fn_hr,
                    ff_hr,
                    hr,
                ],
                dim=1,
            )

            delta = (
                self.band4_refine(
                    b4_in
                )
            )

            alpha = (
                0.25
                * torch.sigmoid(
                    self.alpha_raw
                )
            )

            sr = sr_initial.clone()

            sr[
                :,
                SPECIAL_BAND_INDEX:
                SPECIAL_BAND_INDEX+1,
            ] = (
                sr_initial[
                    :,
                    SPECIAL_BAND_INDEX:
                    SPECIAL_BAND_INDEX+1,
                ]
                + alpha
                * torch.tanh(
                    delta
                )
            )

        else:
            sr = sr_initial

            alpha = torch.zeros(
                (),
                device=lr.device,
            )

        logvar = (
            self.unc_head(
                fused
            )
            .clamp(
                -6.0,
                4.0,
            )
        )

        lr_reconstructed = (
            degradation_operator(
                sr
            )
        )

        return {
            "sr": sr,
            "sr_initial": sr_initial,
            "logvar": logvar,
            "gate": gate,
            "lr_reconstructed": lr_reconstructed,
            "beta": beta,
            "alpha": alpha,
        }


model = MaxSRV2().to(
    DEVICE
)

PARAMS = sum(
    p.numel()
    for p
    in model.parameters()
)

TRAINABLE = sum(
    p.numel()
    for p
    in model.parameters()
    if p.requires_grad
)

print(
    "True mamba_ssm available:",
    HAS_TRUE_MAMBA,
)

if not HAS_TRUE_MAMBA:
    print(
        "Using the included portable PyTorch SSM fallback."
    )

print(
    "Parameters:",
    f"{PARAMS:,}",
)

print(
    "Trainable :",
    f"{TRAINABLE:,}",
)

with torch.no_grad():
    q = torch.randn(
        1,
        4,
        36,
        36,
        device=DEVICE,
    )

    z = model(q)

print(
    "Shape QA:",
    tuple(q.shape),
    "->",
    tuple(z["sr"].shape),
)

assert z["sr"].shape == (
    1, 4, 72, 72
)

assert z["gate"].shape == (
    1, 4, 36, 36
)

assert z["logvar"].shape == (
    1, 4, 72, 72
)

assert z[
    "lr_reconstructed"
].shape == (
    1, 4, 36, 36
)

print("Architecture QA: PASS")


## 6) Initialize the EHFNet spatial backbone from the existing exact-x2 checkpoint

In [ ]:

PRETRAIN_PATH = next(
    (
        p
        for p
        in PRETRAIN_CANDIDATES
        if p.exists()
    ),
    None,
)

PRETRAIN_LOADED = False

if PRETRAIN_PATH is not None:
    raw = torch.load(
        PRETRAIN_PATH,
        map_location="cpu",
    )

    state = raw.get(
        "model_state",
        raw.get(
            "state_dict",
            raw.get(
                "model",
                raw,
            ),
        ),
    )

    backbone_state = (
        model.backbone.state_dict()
    )

    compatible = {}

    for key, value in state.items():
        clean_key = (
            key[7:]
            if key.startswith("module.")
            else key
        )

        if (
            clean_key
            in backbone_state
            and backbone_state[
                clean_key
            ].shape
            == value.shape
        ):
            compatible[
                clean_key
            ] = value

    missing, unexpected = (
        model.backbone.load_state_dict(
            compatible,
            strict=False,
        )
    )

    PRETRAIN_LOADED = (
        len(compatible)
        > 0
    )

    print(
        "Pretrain checkpoint:",
        PRETRAIN_PATH,
    )

    print(
        "Loaded backbone tensors:",
        len(compatible),
        "/",
        len(backbone_state),
    )

    print(
        "Missing backbone keys:",
        len(missing),
    )

else:
    print(
        "No prior exact-x2 EHFNet checkpoint found."
    )

print(
    "PRETRAIN_LOADED:",
    PRETRAIN_LOADED,
)


## 7) Composite objective, EMA, optimizer and staged training

In [ ]:

def gate_balance_loss(
    gate,
):
    usage = gate.mean(
        dim=(0, 2, 3)
    )

    target = torch.full_like(
        usage,
        1.0 / usage.numel(),
    )

    return (
        (usage - target)**2
    ).mean()


def composite_loss(
    outputs,
    gt,
    lr,
    epoch,
):
    sr = outputs["sr"]
    logvar = outputs["logvar"]

    losses = {}

    losses["pixel"] = (
        charbonnier(
            sr - gt
        )
    )

    losses["band4"] = (
        charbonnier(
            sr[
                :,
                SPECIAL_BAND_INDEX:
                SPECIAL_BAND_INDEX+1,
            ]
            -
            gt[
                :,
                SPECIAL_BAND_INDEX:
                SPECIAL_BAND_INDEX+1,
            ]
        )
    )

    b4_error = (
        sr[
            :,
            SPECIAL_BAND_INDEX:
            SPECIAL_BAND_INDEX+1,
        ]
        -
        gt[
            :,
            SPECIAL_BAND_INDEX:
            SPECIAL_BAND_INDEX+1,
        ]
    )

    losses["bias"] = (
        b4_error.mean().abs()
    )

    losses["sam"] = sam_loss(
        sr,
        gt,
    )

    losses["inter"] = (
        interband_loss(
            sr,
            gt,
        )
    )

    losses["edge"] = edge_loss(
        sr,
        gt,
    )

    losses["freq"] = (
        frequency_loss(
            sr,
            gt,
        )
    )

    losses["dc"] = (
        charbonnier(
            outputs[
                "lr_reconstructed"
            ]
            - lr
        )
    )

    losses["gate"] = (
        gate_balance_loss(
            outputs["gate"]
        )
    )

    err2 = (
        sr - gt
    ).pow(2)

    losses["unc"] = (
        0.5
        * torch.exp(
            -logvar
        )
        * err2
        + 0.5
        * logvar
    ).mean()

    if ENABLE_NDVI_LOSS:
        losses["ndvi"] = (
            ndvi_loss(
                sr,
                gt,
            )
        )
    else:
        losses["ndvi"] = (
            sr.new_zeros(())
        )

    with torch.no_grad():
        losses[
            "band4_rmse"
        ] = torch.sqrt(
            (
                b4_error
                * b4_error
            ).mean()
        )

    # Physical consistency ramps up over the first 30 epochs.
    dc_ramp = min(
        1.0,
        max(
            0.0,
            epoch / 30.0,
        ),
    )

    total = losses["pixel"]

    controller = (
        model.loss_weight_controller
    )

    for name, base_weight in (
        BASE_LOSS_WEIGHTS.items()
    ):
        if base_weight == 0:
            continue

        weight = (
            base_weight
            * controller.multiplier(
                name
            )
        )

        if name == "dc":
            weight = (
                weight
                * dc_ramp
            )

        total = (
            total
            + weight
            * losses[name]
        )

    total = (
        total
        + 1e-4
        * controller.regularizer()
    )

    losses["total"] = total

    return losses


class EMA:
    def __init__(
        self,
        model,
        decay=0.9995,
    ):
        self.decay = decay

        self.shadow = {
            key: value.detach().clone()
            for key, value
            in model.state_dict().items()
        }

    @torch.no_grad()
    def update(
        self,
        model,
    ):
        current = (
            model.state_dict()
        )

        for key, value in current.items():
            if torch.is_floating_point(
                value
            ):
                self.shadow[
                    key
                ].mul_(
                    self.decay
                ).add_(
                    value.detach(),
                    alpha=1.0
                    - self.decay,
                )
            else:
                self.shadow[
                    key
                ].copy_(
                    value
                )

    def state_dict(self):
        return {
            key: value.cpu()
            for key, value
            in self.shadow.items()
        }

    def load_state_dict(
        self,
        state,
    ):
        self.shadow = {
            key: value.to(
                DEVICE
            )
            for key, value
            in state.items()
        }

    def apply_to(
        self,
        model,
    ):
        backup = {
            key: value.detach().clone()
            for key, value
            in model.state_dict().items()
        }

        model.load_state_dict(
            self.shadow,
            strict=True,
        )

        return backup

    def restore(
        self,
        model,
        backup,
    ):
        model.load_state_dict(
            backup,
            strict=True,
        )


ema = EMA(
    model,
    EMA_DECAY,
)

backbone_params = list(
    model.backbone.parameters()
)

backbone_ids = {
    id(p)
    for p
    in backbone_params
}

new_params = [
    p
    for p
    in model.parameters()
    if id(p)
    not in backbone_ids
]

optimizer = torch.optim.AdamW(
    [
        {
            "params": backbone_params,
            "lr": BACKBONE_LR,
        },
        {
            "params": new_params,
            "lr": BASE_LR,
        },
    ],
    weight_decay=WEIGHT_DECAY,
)


def lr_factor(
    epoch_index,
):
    epoch = (
        epoch_index
        + 1
    )

    if epoch <= WARMUP_EPOCHS:
        return max(
            0.1,
            epoch
            / WARMUP_EPOCHS,
        )

    t = (
        epoch
        - WARMUP_EPOCHS
    ) / max(
        1,
        EPOCHS
        - WARMUP_EPOCHS,
    )

    return (
        MIN_LR_RATIO
        + (
            1.0
            - MIN_LR_RATIO
        )
        * 0.5
        * (
            1.0
            + math.cos(
                math.pi
                * t
            )
        )
    )


scheduler = (
    torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lr_factor,
    )
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)


def set_backbone_trainable(
    flag,
):
    for p in model.backbone.parameters():
        p.requires_grad = flag


def maybe_stage_backbone(
    epoch,
):
    if (
        PRETRAIN_LOADED
        and epoch
        <= BACKBONE_FREEZE_EPOCHS
    ):
        set_backbone_trainable(
            False
        )
    else:
        set_backbone_trainable(
            True
        )


LOSS_KEYS = [
    "total",
    "pixel",
    "band4",
    "band4_rmse",
    "bias",
    "sam",
    "inter",
    "edge",
    "freq",
    "dc",
    "unc",
    "gate",
    "ndvi",
]


@torch.no_grad()
def validate(
    epoch,
    use_ema=True,
):
    backup = None

    if use_ema:
        backup = ema.apply_to(
            model
        )

    model.eval()

    sums = {
        key: 0.0
        for key
        in LOSS_KEYS
    }

    count = 0

    for batch in val_loader:
        lr = batch[
            "lr"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        gt = batch[
            "gt"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            outputs = model(lr)

        losses = composite_loss(
            outputs,
            gt.float(),
            lr.float(),
            epoch,
        )

        bs = lr.size(0)

        for key in LOSS_KEYS:
            sums[key] += (
                float(
                    losses[
                        key
                    ].detach().item()
                )
                * bs
            )

        count += bs

    if backup is not None:
        ema.restore(
            model,
            backup,
        )

    return {
        key: value / count
        for key, value
        in sums.items()
    }


def save_checkpoint(
    path,
    epoch,
    metrics,
    history,
):
    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "ema_state": ema.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "metrics": metrics,
            "history": history,
            "config": {
                "BACKBONE_C": BACKBONE_C,
                "FUSION_C": FUSION_C,
                "transformer_blocks": NUM_TRANSFORMER_BLOCKS,
                "mamba_blocks": NUM_MAMBA_BLOCKS,
                "true_mamba": HAS_TRUE_MAMBA,
                "special_band_index": SPECIAL_BAND_INDEX,
            },
        },
        path,
    )


## 8) Train — 1000 epochs — auto-resume

In [ ]:

history = []
start_epoch = 1

best_total = float(
    "inf"
)

best_pixel = float(
    "inf"
)

best_band4 = float(
    "inf"
)

if (
    AUTO_RESUME
    and RESUME.exists()
):
    state = torch.load(
        RESUME,
        map_location=DEVICE,
    )

    model.load_state_dict(
        state[
            "model_state"
        ],
        strict=True,
    )

    if "ema_state" in state:
        ema.load_state_dict(
            state[
                "ema_state"
            ]
        )

    optimizer.load_state_dict(
        state[
            "optimizer"
        ]
    )

    scheduler.load_state_dict(
        state[
            "scheduler"
        ]
    )

    if "scaler" in state:
        scaler.load_state_dict(
            state[
                "scaler"
            ]
        )

    history = state.get(
        "history",
        [],
    )

    start_epoch = (
        int(
            state[
                "epoch"
            ]
        )
        + 1
    )

    if history:
        best_total = min(
            row[
                "val_total"
            ]
            for row
            in history
        )

        best_pixel = min(
            row[
                "val_pixel"
            ]
            for row
            in history
        )

        best_band4 = min(
            row[
                "val_band4_rmse"
            ]
            for row
            in history
        )

    print(
        "Resuming from epoch:",
        start_epoch,
    )


for epoch in range(
    start_epoch,
    EPOCHS + 1,
):
    t0 = time.time()

    maybe_stage_backbone(
        epoch
    )

    model.train()

    sums = {
        key: 0.0
        for key
        in LOSS_KEYS
    }

    count = 0

    optimizer.zero_grad(
        set_to_none=True
    )

    bar = tqdm(
        train_loader,
        desc=(
            f"Epoch "
            f"{epoch:04d}/"
            f"{EPOCHS}"
        ),
    )

    for step, batch in enumerate(
        bar,
        start=1,
    ):
        lr = batch[
            "lr"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        gt = batch[
            "gt"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            outputs = model(lr)

            losses = composite_loss(
                outputs,
                gt.float(),
                lr.float(),
                epoch,
            )

            loss_for_backward = (
                losses[
                    "total"
                ]
                / ACCUM_STEPS
            )

        scaler.scale(
            loss_for_backward
        ).backward()

        do_step = (
            step
            % ACCUM_STEPS
            == 0
            or step
            == len(train_loader)
        )

        if do_step:
            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP,
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            optimizer.zero_grad(
                set_to_none=True
            )

            ema.update(
                model
            )

        bs = lr.size(0)

        for key in LOSS_KEYS:
            sums[key] += (
                float(
                    losses[
                        key
                    ].detach().item()
                )
                * bs
            )

        count += bs

        bar.set_postfix(
            total=(
                f"{sums['total']/count:.4f}"
            ),
            pix=(
                f"{sums['pixel']/count:.4f}"
            ),
            b4=(
                f"{sums['band4_rmse']/count:.4f}"
            ),
        )

    scheduler.step()

    train_metrics = {
        key: value / count
        for key, value
        in sums.items()
    }

    val = validate(
        epoch,
        use_ema=True,
    )

    row = {
        "epoch": epoch,
        "seconds": (
            time.time()
            - t0
        ),
        "lr_backbone": (
            optimizer
            .param_groups[0][
                "lr"
            ]
        ),
        "lr_new": (
            optimizer
            .param_groups[1][
                "lr"
            ]
        ),
        "backbone_trainable": bool(
            any(
                p.requires_grad
                for p
                in model.backbone.parameters()
            )
        ),
        "train_total": train_metrics[
            "total"
        ],
        "train_pixel": train_metrics[
            "pixel"
        ],
        "train_band4_rmse": train_metrics[
            "band4_rmse"
        ],
        "val_total": val[
            "total"
        ],
        "val_pixel": val[
            "pixel"
        ],
        "val_band4_rmse": val[
            "band4_rmse"
        ],
        "val_sam": val[
            "sam"
        ],
        "val_dc": val[
            "dc"
        ],
        "val_edge": val[
            "edge"
        ],
        "val_freq": val[
            "freq"
        ],
    }

    history.append(
        row
    )

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY,
        index=False,
    )

    if val["total"] < best_total:
        best_total = val[
            "total"
        ]

        save_checkpoint(
            BEST_TOTAL,
            epoch,
            val,
            history,
        )

    if val["pixel"] < best_pixel:
        best_pixel = val[
            "pixel"
        ]

        save_checkpoint(
            BEST_PIXEL,
            epoch,
            val,
            history,
        )

    if (
        val["band4_rmse"]
        < best_band4
    ):
        best_band4 = val[
            "band4_rmse"
        ]

        save_checkpoint(
            BEST_BAND4,
            epoch,
            val,
            history,
        )

    save_checkpoint(
        RESUME,
        epoch,
        val,
        history,
    )

    if (
        epoch
        % PERIODIC_EVERY
        == 0
    ):
        save_checkpoint(
            CKPT
            / f"epoch_{epoch:04d}.pt",
            epoch,
            val,
            history,
        )

    print(
        f"Epoch {epoch:04d} | "
        f"Val total {val['total']:.6f} | "
        f"Pixel {val['pixel']:.6f} | "
        f"B4 RMSE {val['band4_rmse']:.6f} | "
        f"SAM {val['sam']:.6f} | "
        f"DC {val['dc']:.6f}"
    )


final_val = validate(
    EPOCHS,
    use_ema=True,
)

save_checkpoint(
    LAST,
    EPOCHS,
    final_val,
    history,
)

print("TRAINING COMPLETE")
print(
    "Best val total      :",
    best_total,
)
print(
    "Best val pixel      :",
    best_pixel,
)
print(
    "Best Band-4 val RMSE:",
    best_band4,
)


## 9) Training curves + x8 TTA + best-checkpoint ensemble

In [ ]:

hist = pd.read_csv(
    HISTORY
)

plt.figure(
    figsize=(12, 5)
)

plt.plot(
    hist.epoch,
    hist.val_total,
    label="Val total",
)

plt.plot(
    hist.epoch,
    hist.val_pixel,
    label="Val pixel",
)

plt.plot(
    hist.epoch,
    hist.val_band4_rmse,
    label="Val Band-4 RMSE",
)

plt.xlabel("Epoch")
plt.ylabel(
    "Normalized validation metric"
)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

curve_path = (
    FIG
    / "training_curves.png"
)

plt.savefig(
    curve_path,
    dpi=180,
    bbox_inches="tight",
)

plt.show()


def tta_apply(
    x,
    k,
    flip,
):
    y = torch.rot90(
        x,
        k,
        dims=(-2, -1),
    )

    if flip:
        y = torch.flip(
            y,
            dims=(-1,),
        )

    return y


def tta_invert(
    y,
    k,
    flip,
):
    if flip:
        y = torch.flip(
            y,
            dims=(-1,),
        )

    return torch.rot90(
        y,
        -k,
        dims=(-2, -1),
    )


@torch.no_grad()
def predict_tta(
    model,
    x,
    use_tta=True,
):
    if not use_tta:
        return model(x)["sr"]

    predictions = []

    for k in range(4):
        for flip in (
            False,
            True,
        ):
            xt = tta_apply(
                x,
                k,
                flip,
            )

            yt = model(
                xt
            )["sr"]

            predictions.append(
                tta_invert(
                    yt,
                    k,
                    flip,
                )
            )

    return torch.stack(
        predictions,
        dim=0,
    ).mean(0)


def load_inference_state(
    path,
):
    state = torch.load(
        path,
        map_location=DEVICE,
    )

    if "ema_state" in state:
        model.load_state_dict(
            state[
                "ema_state"
            ],
            strict=True,
        )

        kind = "EMA"

    else:
        model.load_state_dict(
            state[
                "model_state"
            ],
            strict=True,
        )

        kind = "raw"

    model.eval()

    return (
        int(
            state[
                "epoch"
            ]
        ),
        kind,
    )


ensemble_paths = [
    p
    for p
    in [
        BEST_TOTAL,
        BEST_PIXEL,
        BEST_BAND4,
    ]
    if p.exists()
]

if not ensemble_paths:
    raise RuntimeError(
        "No trained best checkpoint was found."
    )

if not USE_CHECKPOINT_ENSEMBLE:
    ensemble_paths = [
        BEST_TOTAL
        if BEST_TOTAL.exists()
        else ensemble_paths[0]
    ]

print(
    "Ensemble checkpoints:"
)

for p in ensemble_paths:
    print(" -", p)


@torch.no_grad()
def ensemble_predict_batch(
    lr,
):
    outputs = []

    for p in ensemble_paths:
        load_inference_state(
            p
        )

        outputs.append(
            predict_tta(
                model,
                lr,
                use_tta=USE_TTA_X8,
            )
        )

    return torch.stack(
        outputs,
        dim=0,
    ).mean(0)


def sam_numpy(
    pred,
    gt,
    eps=1e-12,
):
    dot = np.sum(
        pred * gt,
        axis=1,
    )

    pn = np.sqrt(
        np.sum(
            pred * pred,
            axis=1,
        )
    )

    gn = np.sqrt(
        np.sum(
            gt * gt,
            axis=1,
        )
    )

    cos = np.clip(
        dot
        / np.maximum(
            pn * gn,
            eps,
        ),
        -1.0,
        1.0,
    )

    return float(
        np.degrees(
            np.arccos(
                cos
            )
        ).mean()
    )


pred_batches = []
ref_batches = []

for batch in tqdm(
    test_loader,
    desc="Test ensemble + TTA",
):
    lr = batch[
        "lr"
    ].to(
        DEVICE,
        non_blocking=True,
    )

    gt_norm = batch[
        "gt"
    ].numpy()

    pred_norm = (
        ensemble_predict_batch(
            lr
        )
        .cpu()
        .numpy()
    )

    pred_dn = (
        pred_norm
        * BAND_STD[None]
        + BAND_MEAN[None]
    )

    ref_dn = (
        gt_norm
        * BAND_STD[None]
        + BAND_MEAN[None]
    )

    pred_batches.append(
        pred_dn.astype(
            np.float32
        )
    )

    ref_batches.append(
        ref_dn.astype(
            np.float32
        )
    )

pred = np.concatenate(
    pred_batches,
    axis=0,
)

ref = np.concatenate(
    ref_batches,
    axis=0,
)

error = (
    pred
    - ref
)

mae = float(
    np.mean(
        np.abs(error)
    )
)

rmse = float(
    np.sqrt(
        np.mean(
            error**2
        )
    )
)

b4_error = error[
    :,
    SPECIAL_BAND_INDEX,
]

b4_mae = float(
    np.mean(
        np.abs(
            b4_error
        )
    )
)

b4_rmse = float(
    np.sqrt(
        np.mean(
            b4_error**2
        )
    )
)

b4_bias = float(
    np.mean(
        b4_error
    )
)

per_band = []
ssims = []
psnrs = []

for b in range(4):
    e = error[
        :,
        b,
    ]

    band_range = max(
        float(
            TRAIN_RANGE[b]
        ),
        1e-12,
    )

    band_rmse = float(
        np.sqrt(
            np.mean(
                e**2
            )
        )
    )

    psnr = float(
        20.0
        * np.log10(
            band_range
            / max(
                band_rmse,
                1e-12,
            )
        )
    )

    psnrs.append(
        psnr
    )

    values = []

    for i in range(
        len(pred)
    ):
        values.append(
            skimage_ssim(
                ref[
                    i,
                    b,
                ],
                pred[
                    i,
                    b,
                ],
                data_range=band_range,
            )
        )

    ssim = float(
        np.mean(
            values
        )
    )

    ssims.append(
        ssim
    )

    per_band.append(
        {
            "band": b + 1,
            "MAE": float(
                np.mean(
                    np.abs(e)
                )
            ),
            "RMSE": band_rmse,
            "Bias": float(
                np.mean(e)
            ),
            "PSNR": psnr,
            "SSIM": ssim,
        }
    )


metrics = {
    "ensemble_checkpoints": [
        str(p)
        for p
        in ensemble_paths
    ],
    "tta_x8": USE_TTA_X8,
    "MAE": mae,
    "RMSE": rmse,
    "PSNR_mean": float(
        np.mean(
            psnrs
        )
    ),
    "SSIM_mean": float(
        np.mean(
            ssims
        )
    ),
    "SAM_mean_deg": sam_numpy(
        pred,
        ref,
    ),
    "Band4_MAE": b4_mae,
    "Band4_RMSE": b4_rmse,
    "Band4_Bias": b4_bias,
    "per_band": per_band,
}

TEST_JSON.write_text(
    json.dumps(
        metrics,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        metrics,
        indent=2,
    )
)

print(
    "Saved:",
    TEST_JSON,
)


## 10) Optional full-scene Float32 GeoTIFF export

In [ ]:

if RUN_FULL_SCENE_EXPORT:
    assert LR_PATH.exists()
    assert GT_PATH.exists()

    export_dir = (
        RUN
        / "full_scene_export"
    )

    export_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    out_tif = (
        export_dir
        / "GRAMM_MAXSR_V2_1m_Float32.tif"
    )

    with rasterio.open(
        LR_PATH
    ) as ds:
        lr_full = (
            ds.read()
            .astype(
                np.float32
            )
        )

    with rasterio.open(
        GT_PATH
    ) as ds:
        gt_profile = (
            ds.profile.copy()
        )

        out_h = ds.height
        out_w = ds.width

    lr_norm = (
        lr_full
        - BAND_MEAN
    ) / BAND_STD

    _, H, W = (
        lr_norm.shape
    )

    accum = np.zeros(
        (
            4,
            out_h,
            out_w,
        ),
        dtype=np.float32,
    )

    weight = np.zeros(
        (
            1,
            out_h,
            out_w,
        ),
        dtype=np.float32,
    )

    tile = (
        FULL_SCENE_LR_TILE
    )

    stride = (
        FULL_SCENE_LR_STRIDE
    )

    hann_y = np.hanning(
        tile * 2
    ).astype(
        np.float32
    )

    hann_x = np.hanning(
        tile * 2
    ).astype(
        np.float32
    )

    window = np.outer(
        hann_y,
        hann_x,
    )

    window = np.maximum(
        window,
        1e-3,
    )[None]


    def starts(
        n,
        tile,
        stride,
    ):
        positions = list(
            range(
                0,
                max(
                    1,
                    n - tile + 1,
                ),
                stride,
            )
        )

        last = max(
            0,
            n - tile,
        )

        if (
            not positions
            or positions[-1]
            != last
        ):
            positions.append(
                last
            )

        return positions


    ys = starts(
        H,
        tile,
        stride,
    )

    xs = starts(
        W,
        tile,
        stride,
    )

    for y in tqdm(
        ys,
        desc="Full scene rows",
    ):
        for x in xs:
            patch = lr_norm[
                :,
                y:y+tile,
                x:x+tile,
            ]

            if patch.shape[
                -2:
            ] != (
                tile,
                tile,
            ):
                continue

            inp = torch.from_numpy(
                patch[None]
            ).to(
                DEVICE
            )

            outputs = []

            for p in ensemble_paths:
                load_inference_state(
                    p
                )

                outputs.append(
                    predict_tta(
                        model,
                        inp,
                        use_tta=(
                            FULL_SCENE_USE_TTA
                        ),
                    )
                )

            prediction = (
                torch.stack(
                    outputs,
                    dim=0,
                )
                .mean(0)[0]
                .cpu()
                .numpy()
            )

            prediction = (
                prediction
                * BAND_STD
                + BAND_MEAN
            )

            oy = y * 2
            ox = x * 2

            accum[
                :,
                oy:oy+tile*2,
                ox:ox+tile*2,
            ] += (
                prediction
                * window
            )

            weight[
                :,
                oy:oy+tile*2,
                ox:ox+tile*2,
            ] += window

    sr_full = (
        accum
        / np.maximum(
            weight,
            1e-8,
        )
    )

    profile = (
        gt_profile.copy()
    )

    profile.update(
        driver="GTiff",
        count=4,
        dtype="float32",
        compress="deflate",
        predictor=3,
        BIGTIFF="IF_SAFER",
    )

    with rasterio.open(
        out_tif,
        "w",
        **profile,
    ) as dst:
        dst.write(
            sr_full.astype(
                np.float32
            )
        )

    print(
        "Saved full-scene SR:",
        out_tif,
    )

else:
    print(
        "Full-scene export skipped. "
        "Set RUN_FULL_SCENE_EXPORT=True after training to enable it."
    )


## 11) Final configuration and output paths

In [ ]:

config = {
    "task": (
        "GRAMM 4-band exact x2 "
        "2m -> 1m"
    ),
    "input": [
        4,
        36,
        36,
    ],
    "output": [
        4,
        72,
        72,
    ],
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "accum_steps": ACCUM_STEPS,
    "backbone_c": BACKBONE_C,
    "fusion_c": FUSION_C,
    "transformer_blocks": (
        NUM_TRANSFORMER_BLOCKS
    ),
    "mamba_blocks": (
        NUM_MAMBA_BLOCKS
    ),
    "true_mamba_available": (
        HAS_TRUE_MAMBA
    ),
    "pretrained_backbone_loaded": (
        PRETRAIN_LOADED
    ),
    "pretrain_path": (
        str(PRETRAIN_PATH)
        if PRETRAIN_PATH
        else None
    ),
    "special_band_index_zero_based": (
        SPECIAL_BAND_INDEX
    ),
    "ndvi_loss_enabled": (
        ENABLE_NDVI_LOSS
    ),
    "ema_decay": EMA_DECAY,
    "tta_x8": USE_TTA_X8,
    "checkpoint_ensemble": (
        USE_CHECKPOINT_ENSEMBLE
    ),
    "mtf_nyquist_assumption": (
        LR_TOTAL_MTF_NYQ
    ),
}

CONFIG_JSON.write_text(
    json.dumps(
        config,
        indent=2,
    ),
    encoding="utf-8",
)

print("=" * 100)
print("MAX-SR V2 LOCAL PIPELINE")
print("=" * 100)
print("Run folder   :", RUN)
print("Best total   :", BEST_TOTAL)
print("Best pixel   :", BEST_PIXEL)
print("Best Band-4  :", BEST_BAND4)
print("Resume       :", RESUME)
print("History      :", HISTORY)
print("Test metrics :", TEST_JSON)
print("Config       :", CONFIG_JSON)
print("=" * 100)
